# 🤖 Classify with your own LLM
### A small language model on your own Colab GPU, on the posts you know from Friday

**Module 2 · Session 02 · notebook 1 of 3**

On Friday you classified 457 advice-forum posts. The best model reached 88.7 % on the test posts, and it needed
342 labelled posts to learn from. Today we try the same job with no training data at all: we describe the
categories in words and let a language model read each post.

A **large language model (LLM)** is a neural network trained to continue text. Asked "which category is this
post?", it continues with an answer. The model we use today, `gemma4:e2b`, is small enough to run on the free
GPU that Colab gives you. It runs through **Ollama**, a program that downloads language models and serves them on
your own machine. Nothing is sent to an outside company, and you need no account and no key.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.
The first cell takes a few minutes. The first call to a model can take up to three minutes while it loads. After that it is fast.
If Colab asks for access to `HF_TOKEN`, click *Cancel*.

**Contents**
1. Talk to the model
2. Classify with label names only (zero-shot)
3. Better label definitions
4. Add examples (few-shot)
5. ✏️ Challenge: beat Friday
6. Read the mistakes
7. A harder job with the same code
8. The recipe

**Own machine.** This notebook also runs in Jupyter or VS Code on a laptop with Ollama installed: Moodle explains how. Leave `BACKEND` on `ollama`.

**Plan B, if local does not work** (no free GPU, or Ollama fails). Create a free key at [aistudio.google.com](https://aistudio.google.com) and
store it as a Colab secret called `GEMINI_API_KEY` (key icon on the left, then switch on access for this notebook). In the setup cell, set `BACKEND`
to `gemini` and run everything again. A Groq key from [console.groq.com](https://console.groq.com), stored as `GROQ_API_KEY`, works the same way with `groq`.

The code stays the same and your texts go to the provider. Free keys allow few calls a minute, so plan B runs on smaller samples and takes longer. Your numbers will differ from the slides, because the cloud model is stronger.

In [ ]:
# @title Setup: install Ollama and download the models (2 to 4 minutes) { display-mode: "form" }
BACKEND = "ollama"  # @param ["ollama", "gemini", "groq"]
%pip install -q -U openai datasets
import hashlib, json, os, random, shutil, subprocess, sys, time, warnings
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score
warnings.filterwarnings("ignore")

OLLAMA = "http://localhost:11434"
DECIDER = "d3:2b"
DEVICE = "cpu" if sys.platform == "darwin" else None   # on a Mac, Ollama already fills the GPU memory
if BACKEND == "ollama":
    if shutil.which("ollama") is None and "google.colab" in sys.modules:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed. On your own machine, install it as described on Moodle. Otherwise use plan B: see the top of this notebook.")

    # start_new_session keeps the server alive when you stop a cell
    subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    for model in ["gemma4:e2b"]:
        t0 = time.time()
        done = subprocess.run(["ollama", "pull", model], capture_output=True)
        print(f"{model} ready ({time.time() - t0:.0f} s)" if done.returncode == 0 else f"{model} did not download. Use plan B: see the top of this notebook.")

    if "google.colab" in sys.modules:
        gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
        print(gpu if gpu else "No GPU on this runtime. Each text would take 10 to 20 seconds, which is too slow here. Use Runtime → Change runtime type → T4 GPU. If Colab has none free, use plan B: see the top of this notebook.")
    BASE_URL, API_KEY, MODEL, EXTRA, PAUSE = OLLAMA + "/v1", "ollama", "gemma4:e2b", {"reasoning_effort": "none"}, 0
else:   # plan B: the same calls go to a cloud model. Free keys allow 15 (Gemini) or 30 (Groq) calls a minute
    BASE_URL, secret, MODEL, EXTRA, PAUSE = {
        "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY", "gemini-flash-lite-latest", {}, 4.2),
        "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-20b", {"reasoning_effort": "low"}, 3),
    }[BACKEND]
    try:
        from google.colab import userdata
        API_KEY = userdata.get(secret)
    except ImportError:
        API_KEY = os.environ[secret]

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 200)

---
# 💬 1 · Talk to the model

We talk to the model through the `openai` Python package. The package is named after one company, but most providers use the
same format, and Ollama does too. So the code you write today also works with a cloud provider later: you only change the web
address in `base_url`. Notebook 2 does exactly that. The setup cell stored the address of your model in `BASE_URL`.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(BASE_URL, "·", MODEL)

Every call has two messages. The **system** message sets the job. The **user** message is the text to work on.

In [ ]:
system = "You explain things to business students in plain words."
user = "What is TF-IDF? Three short sentences."

response = client.chat.completions.create(
    model=MODEL,
    temperature=0.3,
    messages=[{"role": "system", "content": system},
              {"role": "user", "content": user}],
    **EXTRA,   # switches off the model's thinking step, which we do not need
)

print(response.choices[0].message.content)
print(response.usage.completion_tokens, "tokens")

✏️ *Change the system message so the model answers as a sceptical investor. Then set `temperature=1.2` and run it
twice. **Temperature** controls how much chance goes into each word. For repeatable work we use 0.*

---
# 📬 2 · Classify with label names only

**Zero-shot** means no examples: the model gets an instruction and nothing else.

Same data and same test posts as Friday.

In [ ]:
df = pd.read_json("https://raw.githubusercontent.com/aaubs/ds-master/main/data/dk_posts_synth_en_processed.json")
df["text"] = df["title"] + ". " + df["selftext"]
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["label"])
if BACKEND != "ollama":
    test = test.head(40)   # plan B: free keys allow few calls a minute

LABELS = sorted(df["label"].unique())
print(len(test), "test posts")
print(LABELS)

In [ ]:
post = test["text"].iloc[0]
print(post[:400])
print("\nLabel given by a person:", test["label"].iloc[0])

First the plain way: we just ask.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{"role": "system", "content": "Classify this post to an advice column."},
              {"role": "user", "content": post}],
    **EXTRA,
)

print(response.choices[0].message.content)

A person can read that answer. A program cannot count it. To get an answer that code can use, we describe the answer as a small
class, using **Pydantic**, a Python library for describing data. `Literal` lists the only allowed values, so the model cannot invent a category.

In [ ]:
class Post(BaseModel):
    category: Literal[tuple(LABELS)]

print(json.dumps(Post.model_json_schema(), indent=2)[:500])

That printout is the **schema**, the form of the answer. We send it along with the request. Two things change in the call: `parse` replaces
`create`, and `response_format` names our class. The answer comes back as an object of that class, in `.parsed`.

In [ ]:
response = client.chat.completions.parse(
    model=MODEL,
    temperature=0,
    messages=[{"role": "system", "content": "Classify this post to an advice column."},
              {"role": "user", "content": post}],
    response_format=Post,
    **EXTRA,
)

answer = response.choices[0].message.parsed
print(answer)
print(answer.category)

Did it pick a sensible category? We only said "classify". The schema limits what the model may answer, but the schema is not part of what it reads.

Now five posts, with a plain loop.

In [ ]:
for text, label in zip(test["text"][:5], test["label"][:5]):
    response = client.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[{"role": "system", "content": "Classify this post to an advice column."},
                  {"role": "user", "content": text}],
        response_format=Post,
        **EXTRA,
    )
    print(f"{response.choices[0].message.parsed.category:<45} a person said: {label}")

We have now written the same call three times, and only the system message, the text and the class change. That is the moment for a
function. `ask()` is the same call with those three things as inputs.

In [ ]:
def ask(system, user, schema):
    response = client.chat.completions.parse(
        model=MODEL,
        temperature=0,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        response_format=schema,
        **EXTRA,
    )
    return response.choices[0].message.parsed

ask("Classify this post to an advice column.", post, Post)

On one post that is enough. On hundreds, a call sometimes fails, and your loop stops at row 80. So we add a retry. It raises the temperature a
little each time, because the same temperature would repeat the same mistake. `PAUSE` is 0 on Ollama: it only matters for plan B.

This is the version we use from now on.

In [ ]:
def ask(system, user, schema):
    for temperature in (0, 0.2, 0.4):   # the same temperature would repeat the same mistake
        time.sleep(PAUSE)
        try:
            response = client.chat.completions.parse(
                model=MODEL,
                temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                response_format=schema,
                **EXTRA,
            )
            return response.choices[0].message.parsed
        except Exception as e:
            error = e
            if "429" in str(e):
                time.sleep(30)   # over the free limit: wait for the next minute
    raise error

Now all the test posts, still with a plain loop.

In [ ]:
system = "Classify this post to an advice column."

t0 = time.time()
preds = []
for text in test["text"]:
    answer = ask(system, text, Post)
    preds.append(answer.category)

accuracy = accuracy_score(test["label"], preds)
print(f"{accuracy:.1%} in {time.time() - t0:.0f} s")

🤔 *What happened? The schema guarantees the form of the answer. The model still has to be told what the options mean.*

*On plan B this run scores far higher, about 90 %. Gemini seems to read the schema, and Ollama does not. The same code can behave differently on another provider, so you measure again when you switch.*

So we put the labels into the system message. We will run this loop many times today, with only the system message changing, so the loop
becomes a function too. `run()` also keeps every score in `RESULTS`, starting with Friday's two models and the run you just did.

In [ ]:
RESULTS = {"Friday: TF-IDF + SVM, 342 labels": 0.852,
           "Friday: embeddings + SVM, 342 labels": 0.887,
           "LLM: labels not in the prompt": accuracy}

def run(name, system):
    t0 = time.time()
    preds = []
    for text in test["text"]:
        answer = ask(system, text, Post)
        preds.append(answer.category)
    accuracy = accuracy_score(test["label"], preds)
    RESULTS[name] = accuracy
    print(f"{name}: {accuracy:.1%} in {time.time() - t0:.0f} s")
    return preds

system = "Classify this post to an advice column. Categories: " + ", ".join(LABELS)
print(system)
p_names = run("LLM: label names in the prompt", system)

---
# 📝 3 · Better label definitions

The model only knows what the words in the label mean to it. *Practical Questions & Everyday Life* and *Everyday
Observations & Rants* are easy to confuse. In annotation work, the document that explains each label is called a
**codebook**. Here the codebook is simply part of the system message.

In [ ]:
CODEBOOK = {
    "Love & Dating": "Romantic relationships, dating, partners, jealousy, breakups.",
    "Family Dynamics": "Siblings, parents, children, in-laws, family boundaries.",
    "Work, Study & Career": "Jobs, colleagues, bosses, studies, career choices.",
    "Friendship & Social Life": "Friends, social circles, loneliness, fitting in.",
    "Health & Wellness (Physical and Mental)": "Physical or mental health, stress, sleep, diet.",
    "Personal Finance & Housing": "Money, debt, rent, buying a home.",
    "Practical Questions & Everyday Life": "How-to questions, household tasks, small practical things.",
    "Everyday Observations & Rants": "Complaints or remarks about daily life, no real question.",
}

system_book = "Classify this post to an advice column. Categories:\n"
for label, definition in CODEBOOK.items():
    system_book += f"- {label}: {definition}\n"

print(system_book)
p_book = run("LLM: label definitions", system_book)

---
# 🎯 4 · Add examples

**Few-shot** means showing a handful of solved examples inside the prompt. Examples work well when a rule is hard to
put into words. They use up prompt length, and the model may copy their style. We take one example per category,
eight in all, from the training posts, so the test posts stay untouched.

In [ ]:
solved = train.groupby("label").sample(1, random_state=1)

system_few = system_book + "\nExamples:\n"
for row in solved.itertuples():
    system_few += f"Post: {row.text[:300]}\nCategory: {row.label}\n"

print(system_few[:900])
p_few = run("LLM: definitions + 8 examples", system_few)

In [ ]:
pd.Series(RESULTS, name="accuracy").sort_values().to_frame().style.format("{:.1%}")

---
# ✏️ 5 · Challenge: beat Friday

Friday's best was 88.7 %, with 342 labelled posts. Your job: get as close as you can with **no training at all**, only
by improving the instructions. Each try takes about a minute. Write your best score on the board.

Things to try: sharper definitions for the two labels that are confused most; a rule such as "if the writer asks
for advice, it is not a rant"; other examples.

One warning. If you keep tuning until the test score is high, the test posts stop being a fair test, exactly as in
module 1. With more time you would tune on a separate set of posts.

In [ ]:
my_book = dict(CODEBOOK)
# ✏️ edit the definitions, for example:
# my_book["Everyday Observations & Rants"] = "..."

my_system = "Classify this post to an advice column. Categories:\n"
for label, definition in my_book.items():
    my_system += f"- {label}: {definition}\n"
my_system += ""   # ✏️ add your own rules here

p_mine = run("LLM: my instructions", my_system)

---
# 🔍 6 · Read the mistakes

A score tells you how often the model is wrong. Only reading tells you why. People who run LLM systems for a living
say this is the step that pays off most, and it is the one most often skipped.

In [ ]:
errors = test[["title", "selftext", "label"]].copy()
errors["predicted"] = p_book
errors = errors[errors["label"] != errors["predicted"]]
print(len(errors), "mistakes\n")

print(errors.groupby(["label", "predicted"]).size().sort_values(ascending=False).head(6))
errors.sample(min(10, len(errors)), random_state=1)

✏️ *Read ten mistakes and sort each into one of three piles:*
- *the **model** is wrong*
- *the **label** in the data is wrong or debatable*
- *the **category** itself is unclear, and two answers are fair*

*How many are in each pile? If most are in the second and third pile, what does the accuracy number really mean?*

---
# 🌡️ 7 · A harder job with the same code

Sorting posts by topic is an easy job: the words in the text point to the answer. Now we keep the code and change the
job. **Stance detection** asks where the writer stands, for or against something. The data is 100 tweets about climate
change, labelled by people as *favor*, *against* or *none* (`tweet_eval` from Cardiff NLP).

We add two measures. The **baseline** is what you get by always giving the most common answer. **Cohen's kappa**
measures agreement beyond chance: 0 means no better than chance, 1 means perfect agreement.

In [ ]:
from datasets import load_dataset
tweets = load_dataset("cardiffnlp/tweet_eval", "stance_climate", split="test").to_pandas().sample(frac=1, random_state=42).head(100 if BACKEND == "ollama" else 40)
truth = tweets["label"].map({0: "none", 1: "against", 2: "favor"}).tolist()

print(pd.Series(truth).value_counts())
baseline = pd.Series(truth).value_counts(normalize=True).max()
print(f"\nbaseline, always the most common answer: {baseline:.0%}")

In [ ]:
STANCE = {"favor": "Supports action on climate change, directly or by implication.",
          "against": "Opposes action on climate change, or denies or dismisses it.",
          "none": "Takes no clear position."}

class Stance(BaseModel):
    stance: Literal["favor", "against", "none"]

system = "What is the stance of this tweet toward action on climate change?\n"
for label, definition in STANCE.items():
    system += f"- {label}: {definition}\n"

preds = []
for text in tweets["text"]:
    preds.append(ask(system, text, Stance).stance)

flips = sum({a, b} == {"favor", "against"} for a, b in zip(preds, truth))
print(f"accuracy {accuracy_score(truth, preds):.0%} · kappa {cohen_kappa_score(truth, preds):.2f} · favor and against swapped: {flips}")

🤔 *The model is below the baseline, although it reads every tweet. Why can a model that seems so capable lose to
"always say favor"?*

🤔 *Is every mistake equally bad? A flip between favor and against reverses the meaning. A miss only loses information.*

✏️ *Change the `STANCE` definitions and run the cell again. How much can better definitions fix here?*

---
# 🧭 8 · The recipe

Every simple LLM job follows the same steps:

1. Write the instruction and the allowed answers.
2. Try five examples by eye.
3. Hand-label about 50 examples, if the data has no labels yet.
4. Run on those and measure. Compare with a baseline, the score you get by always giving the most common answer.
5. Read the mistakes, then fix the instruction.
6. Run everything, saving as you go.
7. Spot-check the result.

**What you saw:** with no training data, a small local model matched Friday's trained classifiers on an easy task, once the
instructions were good. On a task that needs interpretation, the same code fell below a trivial baseline. A result on one task says little
about the next one, so you measure every time.

**Next:** notebook 2 turns text into tables. Notebook 3 uses models that are faster than an LLM, and combines them.